In [0]:
#from datetime import date, datetime
from pyspark.sql.functions import lit, format_number, coalesce, split, current_timestamp
from pyspark.sql import SparkSession
from pyspark.dbutils import DBUtils
from io import BytesIO
import pandas as pd
import boto3
import json
import os

In [0]:
spark = SparkSession.builder.getOrCreate()
dbutils = DBUtils(spark)

# Function to get AWS credentials from Databricks secrets
def get_aws_credentials(secrets_scope, key_name):
    try:
        # Databricks secrets configuration
   
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        if aws_secret_key is not None:
            print("AWS secret key retrieved successfully.")
        else:
            print("AWS secret key is None. Please check Databricks secrets configuration.")
 
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None

# Function to get AWS credentials from Databricks secrets
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration
 
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
   
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

# Function to connect to AWS and access SSM parameter
def connect_to_aws_and_access_ssm(ssm_key, ssm_client, aws_secret_key):
 
    if aws_secret_key is None:
        print("AWS secret key not available.")
        return
 
    # Access SSM parameter /cfn/a360/reports_output_path
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )
 
    return response['Parameter']['Value']

# Function to get the environment prefix
def get_env_prefix():
    secrets_scope = "aws-databricks"
    aws_keys = get_scope_key_names(secrets_scope)
    aws_key = aws_keys[0] if aws_keys else None
    # Get AWS secret key from Databricks secrets
    aws_secret_key = get_aws_credentials(secrets_scope,aws_key)
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)
 
    S3_client = boto3.client('s3', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)  

    env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix', ssm_client, aws_secret_key)
    return env_prefix

## Get the needed Perimeter and Parameters

In [0]:
env = get_env_prefix()

In [0]:
#display(env)

In [0]:
from datetime import timedelta#`ref_calendar` is loaded into a DataFrame
 
ref_calendar = spark.table(f"`{env}-cdh`.reference.ref_calendar")
 
# Filter for the current date--> will be put to run for the running date
today_date = spark.sql("SELECT current_date() as today").collect()[0]['today']
current_date = (today_date - timedelta(days=2)).strftime('%Y-%m-%d')
#current_date='2025-10-28'#-- for manual run
 
# ---CHECK : If date is not a business day, get the previous one ---
calendar_check = ref_calendar.filter(ref_calendar.CalendarDate == current_date).collect()
if not calendar_check:
    raise ValueError(f"Error: The date {current_date} was not found in reference.ref_calendar.")
 
if calendar_check[0]["IsBusinessDay"] == 0:
    current_date = calendar_check[0]["PreviousBusinessDay"]
    print(f"Initial date was not a business day. Using previous business day: {current_date}")
# ---End of Check---
 
filtered_calendar = ref_calendar.filter(ref_calendar.CalendarDate == current_date)
filtered_calendar.createOrReplaceTempView("filtered_calendar")
 
#display(filtered_calendar)
 
row = filtered_calendar.collect()[0]   # since this is a single date, one row
businessday = row["CalendarDate"]
#display(businessday)
 


In [0]:
import pandas as pd
import os
# 1. Define the path to your file
file_path ="/Volumes/qa-cdh-delivery/coc_letters/input_files"
files=[f.name for f in dbutils.fs.ls(file_path) if f.name.endswith('.xlsx')]
if len(files)==0:
    raise ValueError(f"Error: No files found in {file_path}.")
elif len(files)>1:
    raise ValueError(f"Error: More than one file found in {file_path}. ONLY ONE FILE SHOULD EXISTS IN THIS PATH!")
file_name=files[0]
full_path = f"{file_path}/{file_name}"
# 2. Read into a Pandas DataFrame first
# header=0 implies the first row is the column name. 
# If there is no header, use header=None and assign names manually.
pdf = pd.read_excel(full_path, header=0)
 
# Optional: Clean up column name to be safer for Spark (no spaces)
pdf.columns = ['Account_Number']
 
# 3. Convert to Spark Data/Volumes/qa-cdh-delivery/coc_letters/input_filesFrame
df_accounts = spark.createDataFrame(pdf)
df_accounts.createOrReplaceTempView("df_accounts")
# 4. Display the result
#display(df_accounts)


In [0]:
#Contracts
Contracts=spark.sql(f"""
                     select  distinct agreementkey as ContractNo,exposurekey as accountno
                    from `{env}-cdh`.primary.exposure 
                    inner join df_accounts 
                    on df_accounts.Account_Number=substring(exposurekey,4,50)
                      where BusinessDate = '{businessday}'
                 
                 """)
                 
                    

Contracts.createOrReplaceTempView("Contracts")
#display(Contracts)

## Read Source Tables

In [0]:
primary_exposure=spark.sql(f"""
                        select 
                            distinct exposurekey,cdi,agreementkey,PenaltyRate,interestrate
                            ,portfoliokey ,portfolioname,BusinessDate,ProductKey,partykey
                            from    `{env}-cdh`.primary.exposure 
                            where BusinessDate = '{businessday}'

                            """)

primary_exposure.createOrReplaceTempView("primary_exposure")
#display(primary_exposure)

In [0]:
dexposure = spark.sql(f"""
                            select distinct
                                      Exposurekey
                                    , Agreementkey
                                    , AssetSegment
                                    , PortfolioName
                                    , BusinessDate
                                    ,SettlementBucket
                                    ,HasActiveSettlementFlag
                                    ,portfoliokey,lps
                            from `{env}-cdh`.derivative.dexposure dex
                            where BusinessDate = '{businessday}'
                        """)

dexposure.createOrReplaceTempView("dexposure")

In [0]:
party = spark.sql(f"""
                            select distinct
                                      CDI,
                                   FirstNameGR,LastNameGR
                                   ,FirstFatherNameGR
                                   ,businessdate,partykey,PORTFOLIOKEY,portfolioname
                                    
                            from `{env}-cdh`.primary.party d
                            where BusinessDate = '{businessday}'
                        """)

party.createOrReplaceTempView("party")

In [0]:
product=spark.sql(f"""
                            select distinct
                                      Productkey,L3869Flag,
                                    
                                      BusinessDate


                        from `{env}-cdh`.primary.product
                                      where BusinessDate = '{businessday}'
                                      """)
product.createOrReplaceTempView("product")


In [0]:
profits_Loan_balance=spark.sql(f"""
                            select distinct
                                      BusinessDate,ACCOUNT_NUMBER,
                                      NRM_CAP_BAL       
,NRM_RL_INT_BAL    
,NRM_URL_INT_BAL   
,NRM_EXP_BAL       
,OV_CAP_BAL        
,OV_RL_NRM_INT_BAL 
,OV_RL_PNL_INT_BAL 
,OV_URL_NRM_INT_BAL
,OV_URL_PNL_INT_BAL
,OV_EXP_BAL
,NRM_ACCRUAL_AMN   
,OV_ACCRUAL_AMN
,OV_EXP_DT                                         --Ημερομηνία έναρξης καθυστέρησης
,TOTAL_BALANCE 
,Nrm_Url_Int_Bal ,Ov_Url_Nrm_Int_Bal,
Ov_Cap_Bal ,Ov_Rl_Nrm_Int_Bal ,Ov_Url_Nrm_Int_Bal ,Ov_Exp_Bal




                                        from `p-cdh`.landed.profits_Loan_balance
                                      where BusinessDate ='{businessday}'

                                      """)
profits_Loan_balance.createOrReplaceTempView("profits_Loan_balance")

In [0]:
exposure_profits=spark.sql(f"""
                              select 
                            distinct exposurekey,cdi,agreementkey,PenaltyRate,interestrate
                            ,portfoliokey ,portfolioname,NumberOfInstallments,bucket
                            from   from `{env}-cdh`.primary.exposure 
                            where BusinessDate = '{businessday}' and sourcesystem='profits'

                            """)

exposure_profits.createOrReplaceTempView("exposure_profits")

In [0]:
exposure_qc=spark.sql(f"""
                              select 
                            distinct exposurekey,cdi,agreementkey,PenaltyRate,interestrate
                            ,portfoliokey ,portfolioname,NumberOfInstallments,bucket
                            from   from `{env}-cdh`.primary.exposure 
                            where BusinessDate = '{businessday}' and sourcesystem='qca'

                            """)

exposure_qc.createOrReplaceTempView("exposure_qc")

In [0]:
ActivitiesOrdered=spark.sql(f"""

select  exposurekey,
      InteractionDate,
	  InteractionTypeKey,portfolioname
from `{env}-cdh`.primary.get_interactions_by_business_date('{businessday}')
where interactiontypekey  in ('3100','3101','3102','4019')  """)

ActivitiesOrdered.createOrReplaceTempView("ActivitiesOrdered")
#display(ActivitiesOrdered)



In [0]:
Law4738=spark.sql(f"""


select  cdi,ApplicationStatus,UnderlawType,LawSystem,concat(UnderLawType,'-',LawSystem) as system_4738
from `{env}-cdh`.derivative.dunderlaw

where  businessdate='{businessday}' and underlawtype= 'N4738/2020'

""")

Law4738.createOrReplaceTempView("Law4738")
#display(Law4738)    

In [0]:
Cities=spark.sql(f"""

select  partykey,cdi,postalcode,Street,city,CommunicationFlag
from `{env}-cdh`.primary.contactdetails

where  businessdate='{businessday}' and CommunicationFlag = '1'""")

Cities.createOrReplaceTempView("Cities")
#display(Cities)

In [0]:
realtion_profits=spark.sql(f"""
                           select  distinct 
                           lde.AGREEMENT_NUMBER , lde.prft_acc_unit,
                           lde.ACCOUNT_NUMBER 
                           ,ar.cust_id
                           ,	CASE WHEN  MAIN_BENEF_FLG='1' and Relation_Type='B' 
									THEN  'Agreement-Main Beneficiary Rltp' 
									WHEN Relation_Type='B' then 'Agreement-Other Beneficiaries Rltp'
									WHEN Relation_Type='G' then 'Agreement-Guarantor Rltp'
									END													AS	AssetXPartyTypeId
                           
                             FROM  `p-cdh`.landed.profits_loan_details lde
								INNER JOIN
											`p-cdh`.landed.profits_Agreement_Relation ar
									ON lde.AGREEMENT_NUMBER=ar.agreement_number
                              where  lde.businessdate='{businessday}'    and ar.businessdate='{businessday}' 
                                    """
                                    )
realtion_profits.createOrReplaceTempView("realtion_profits")
#display(realtion_profits)
					

In [0]:
exposure_product=spark.sql(f"""
                         select distinct  ex.exposurekey,cdi,agreementkey,
pqca.SubProduct as subproductcalc
from  `{env}-cdh`.primary.exposure ex 
inner join   `{env}-cdh`.primary.product p
on p.ProductKey=ex.ProductKey and p.BusinessDate='{businessday}' 
left join  `{env}-cdh`.primary.product pqca 
on ex.ProductKey=pqca.ProductKey and pqca.BusinessDate='{businessday}'  and pqca.SourceSystem='qca'
where 1=1   and ex.BusinessDate='{businessday}'  --problem!  

and pqca.SubProduct is not null
""")

exposure_product.createOrReplaceTempView("exposure_product")
#display(exposure_product)

In [0]:
partyrelatedparty=spark.sql(f"""
SELECT DISTINCT RepresentedByPartyKey, PartyRepresentKey,RepresentedByPartyType,LEFT(PartyRepresentKey, CHARINDEX(';', PartyRepresentKey) - 1) AS before_semicolon,partykey,cdi
, SUBSTRING(PartyRepresentKey, CHARINDEX(';', PartyRepresentKey) + 1, LEN(PartyRepresentKey)) AS after_semicolon
FROM  `{env}-cdh`.primary.party
WHERE BusinessDate = '{businessday}'
--AND SourceSystem = 'profits'
and RepresentedByPartyType='REPRESENTS' 

"""
)

partyrelatedparty.createOrReplaceTempView("partyrelatedparty")
#display(partyrelatedparty)

In [0]:
biteam_lpscontract=spark.sql(f"""
WITH portfolio_max_businessdate AS(
  SELECT PortfolioId, MAX(BusinessDate) as MaxBusinessDate
  FROM `{env}-ds`.aaas.lps
  WHERE Granularity = 'Agreement'
  AND BusinessDate <= '{businessday}'
  GROUP BY PortfolioId
)
SELECT DISTINCT
  lps.AgreementKey AS agreementnumber,
  lps.BusinessDate AS businessdate,
  lps.OperationalLegalStatusLevel3Description
FROM `{env}-ds`.aaas.lps lps
JOIN portfolio_max_businessdate maxdate
ON lps.PortfolioId = maxdate.PortfolioId
AND lps.BusinessDate = maxdate.MaxBusinessDate
WHERE lps.Granularity = 'Agreement'
""")
biteam_lpscontract.createOrReplaceTempView("biteam_lpscontract")





# biteam_lpscontract=spark.sql(f"""
# select distinct 
# agreementnumber,businessdate,OperationalLegalStatusLevel3Description
# from  `p-cdh`.landed.biteam_lpscontract 
# where  businessdate='{businessday}' """)
# biteam_lpscontract.createOrReplaceTempView("biteam_lpscontract")

In [0]:
Restructurings=spark.sql(f"""
select distinct NewAgreementKey,businessdate,oldexposurekey,newexposurekey
from  `{env}-cdh`.primary.restructuring 
where  businessdate='{businessday}' """)
Restructurings.createOrReplaceTempView("Restructurings")



Report

In [0]:
fisrt_part =spark.sql(f"""
                     select  distinct 
                           coc.ContractNo,
ex.exposurekey
,party.CDI
,rp.AssetXPartyTypeId relation
,concat(Party.FirstNameGR,' ' ,Party.LastNameGR) as FullName 
,party.FirstFatherNameGR
,Cities.Street
,Cities.City as City
,Cities.PostalCode
,Cities.CommunicationFlag
,ep.subproductcalc  as SubProduct
,Loan_balance.NRM_CAP_BAL                        
,Loan_balance.NRM_RL_INT_BAL                     
,Loan_balance.NRM_URL_INT_BAL                    
,Loan_balance.NRM_EXP_BAL                        
,Loan_balance.OV_CAP_BAL                         
,Loan_balance.OV_RL_NRM_INT_BAL                  
,Loan_balance.OV_RL_PNL_INT_BAL                  
,Loan_balance.OV_URL_NRM_INT_BAL
,Loan_balance.OV_URL_PNL_INT_BAL
,Loan_balance.OV_EXP_BAL
,Loan_balance.NRM_ACCRUAL_AMN     
,Loan_balance.OV_ACCRUAL_AMN
,eprof.InterestRate 
,eprof.PenaltyRate as Penalty_Rate  
,dex.lps as AccountLegalProcessStatus
--,lps.OperationalLegalStatusLevel3Description as AccountLegalProcessStatus
,dex.SettlementBucket
,eprof.NumberOfInstallments
,Loan_balance.OV_EXP_DT                                     
,Loan_balance.TOTAL_BALANCE 
,Loan_balance.Nrm_Url_Int_Bal + Loan_balance.Ov_Url_Nrm_Int_Bal as  `Μη Λογιστικοποιημένοι Τόκοι`
,Loan_balance.Ov_Cap_Bal + Loan_balance.Ov_Rl_Nrm_Int_Bal + Loan_balance.Ov_Url_Nrm_Int_Bal + Loan_balance.Ov_Exp_Bal as  `Ληξιπρόθεσμο Ποσό`
 ,CASE WHEN dex.HasActiveSettlementFlag IS FALSE then 'NO' ELSE 'YES' END as `Ένδειξη settlement (Y/N)`
,coalesce(p.L3869Flag,False) as Law3869Flag
,ex_qc.FirstLetterDate
,ex_qc.SecondLetterDate
,ex_qc.ThirdLetterDate
,ex_qc.CoCStatus  
,act3100.InteractionDate Dateof3100
,act3101.InteractionDate Dateof3101
,act3102.InteractionDate Dateof3102
,act4019.InteractionDate Dateof4019
,case when Restructurings.oldexposurekey <>Restructurings.newexposurekey then 'Yes' else 'No' end as `Ρύθμιση (profits)`
,eprof.bucket SourceBucketId
,case when Law4738.cdi is not null then 'Yes' else 'No' end as `Αίτηση 4738`
,Law4738.ApplicationStatus as `status 4738`
,Law4738.system_4738 as `σύστημα 4738`
                           from Contracts coc
inner join primary_exposure ex
on coc.ContractNo=ex.agreementkey and ex.businessdate='{businessday}' 
inner join profits_Loan_balance loan_balance
on ex.exposurekey=loan_balance.ACCOUNT_NUMBER and loan_balance.businessdate='{businessday}'
inner join  product p
on p.productkey=ex.ProductKey and p.businessdate='{businessday}' 
left join exposure_product ep 
on ep.expoSUREKEY = SUBSTRING(ex.exposurekey,4,50) and ep.cdi=ex.cdi and ep.agreementkey=ex.agreementkey
left join  biteam_lpscontract as  lps 
on lps.agreementnumber=ex.agreementkey and lps.BusinessDate='{businessday}'
left join party 
on ex.partykey=party.partykey and party.businessdate='{businessday}'
 and party.portfoliokey=ex.portfoliokey and party.cdi=ex.cdi
left join primary.exposure  eprof 
on ex.AgreementKey=eprof.AgreementKey and ex.ExposureKey=eprof.exposurekey 
and eprof.businessdate='{businessday}'
 and ex.cdi=eprof.cdi
and eprof.sourcesystem='profits'
left join primary.exposure ex_qc
on ex.agreementkey=ex_qc.agreementkey and SUBSTRING(ex.exposurekey,4,50)=ex_qc.exposurekey and ex_qc.sourcesystem='qca'
and ex_qc.businessdate='{businessday}' 
 and ex_qc.PartyRelationshipTypeCode <>'2666' and  ex_qc.IsActiveFlag=True AND EX.CDI=ex_qc.cdi
left join Cities  on Cities.cdi=party.cdi  
left join ActivitiesOrdered as act3100  on act3100.exposurekey = ex.exposurekey and act3100.InteractionTypeKey = '3100'
left join  ActivitiesOrdered as act3101 on act3101.exposurekey = ex.exposurekey and act3101.InteractionTypeKey = '3101'
left join  ActivitiesOrdered as act3102 on act3102.exposurekey = ex.exposurekey and act3102.InteractionTypeKey = '3102'
left join  ActivitiesOrdered as act4019 on act4019.exposurekey = ex.exposurekey and act4019.InteractionTypeKey = '4019'
left join Law4738 on Law4738.cdi=party.cdi 

left join  dexposure dex on dex.exposurekey=eprof.exposurekey and dex.portfoliokey=eprof.portfoliokey and dex.agreementkey=eprof.agreementkey and dex.businessdate='{businessday}'

left join   Restructurings on Restructurings.NewAgreementKey=ex.AgreementKey 
and restructurings.businessdate='{businessday}'
left join realtion_profits rp on rp.AGREEMENT_NUMBER=eprof.agreementkey and rp.ACCOUNT_NUMBER=eprof.exposurekey  and eprof.partykey=rp.cust_id and eprof.portfoliokey=rp.prft_acc_unit


 
   """)
   

fisrt_part.createOrReplaceTempView("fisrt_part")
#display(fisrt_part)


In [0]:
second_part =spark.sql(f"""select  distinct 
                           coc.ContractNo,
ex.exposurekey
,party.CDI
,  'REPRESENTS' relation
,concat(Party.FirstNameGR,' ' ,Party.LastNameGR) as FullName 
,party.FirstFatherNameGR
,Cities.Street
,Cities.City as City
,Cities.PostalCode
,Cities.CommunicationFlag
,ep.subproductcalc  as SubProduct
,Loan_balance.NRM_CAP_BAL                        
,Loan_balance.NRM_RL_INT_BAL                     
,Loan_balance.NRM_URL_INT_BAL                    
,Loan_balance.NRM_EXP_BAL                        
,Loan_balance.OV_CAP_BAL                         
,Loan_balance.OV_RL_NRM_INT_BAL                  
,Loan_balance.OV_RL_PNL_INT_BAL                  
,Loan_balance.OV_URL_NRM_INT_BAL
,Loan_balance.OV_URL_PNL_INT_BAL
,Loan_balance.OV_EXP_BAL
,Loan_balance.NRM_ACCRUAL_AMN     
,Loan_balance.OV_ACCRUAL_AMN
,eprof.InterestRate 
,eprof.PenaltyRate as Penalty_Rate  
,dex.lps as AccountLegalProcessStatus
--,lps.OperationalLegalStatusLevel3Description as AccountLegalProcessStatus
,dex.SettlementBucket
,eprof.NumberOfInstallments
,Loan_balance.OV_EXP_DT                                         --Ημερομηνία έναρξης καθυστέρησης
,Loan_balance.TOTAL_BALANCE 
,Loan_balance.Nrm_Url_Int_Bal + Loan_balance.Ov_Url_Nrm_Int_Bal as  `Μη Λογιστικοποιημένοι Τόκοι`
,Loan_balance.Ov_Cap_Bal + Loan_balance.Ov_Rl_Nrm_Int_Bal + Loan_balance.Ov_Url_Nrm_Int_Bal + Loan_balance.Ov_Exp_Bal as  `Ληξιπρόθεσμο Ποσό`
 ,CASE WHEN dex.HasActiveSettlementFlag IS FALSE then 'NO' ELSE 'YES' END as `Ένδειξη settlement (Y/N)`
,coalesce(p.L3869Flag,False) as Law3869Flag
,ex_qc.FirstLetterDate
,ex_qc.SecondLetterDate
,ex_qc.ThirdLetterDate
,ex_qc.CoCStatus  
,act3100.InteractionDate Dateof3100
,act3101.InteractionDate Dateof3101
,act3102.InteractionDate Dateof3102
,act4019.InteractionDate Dateof4019
,case when Restructurings.oldexposurekey <>Restructurings.newexposurekey then 'Yes' else 'No' end as `Ρύθμιση (profits)`
,eprof.bucket SourceBucketId 
,case when Law4738.cdi is not null then 'Yes' else 'No' end as `Αίτηση 4738`
,Law4738.ApplicationStatus as `status 4738`
,Law4738.system_4738 as `σύστημα 4738`
                           from Contracts coc
inner join primary_exposure ex
on coc.ContractNo=ex.agreementkey and ex.businessdate='{businessday}' 
inner join profits_Loan_balance loan_balance
on ex.exposurekey=loan_balance.ACCOUNT_NUMBER and loan_balance.businessdate='{businessday}'
inner join product p
on p.productkey=ex.ProductKey and p.businessdate='{businessday}' 
left join exposure_product ep 
on ep.expoSUREKEY = SUBSTRING(ex.exposurekey,4,50) and ep.cdi=ex.cdi and ep.agreementkey=ex.agreementkey
left join  biteam_lpscontract lps 
on lps.agreementnumber=ex.agreementkey and lps.BusinessDate='{businessday}'
left join primary.exposure  eprof 
on ex.AgreementKey=eprof.AgreementKey and ex.ExposureKey=eprof.exposurekey and eprof.businessdate='{businessday}' and ex.cdi=eprof.cdi
and eprof.sourcesystem='profits'
left join primary.exposure ex_qc
on ex.agreementkey=ex_qc.agreementkey and SUBSTRING(ex.exposurekey,4,50)=ex_qc.exposurekey and ex_qc.sourcesystem='qca'
and ex_qc.businessdate='{businessday}'  and ex_qc.PartyRelationshipTypeCode <>'2666' and  ex_qc.IsActiveFlag=True AND EX.CDI=ex_qc.cdi
left join realtion_profits rp on rp.AGREEMENT_NUMBER=eprof.agreementkey and rp.ACCOUNT_NUMBER=eprof.exposurekey  and eprof.partykey=rp.cust_id and eprof.portfoliokey=rp.prft_acc_unit
inner join partyrelatedparty pa
on pa.partykey=rp.cust_id 
left join party 
on pa.after_semicolon=party.partykey and party.businessdate='{businessday}' and party.portfoliokey=ex.portfoliokey 

left join Cities  on Cities.cdi=party.cdi  --and cities.partykey=party.partykey
left join ActivitiesOrdered as act3100  on act3100.exposurekey = ex.exposurekey and act3100.InteractionTypeKey = '3100'
left join  ActivitiesOrdered as act3101 on act3101.exposurekey = ex.exposurekey and act3101.InteractionTypeKey = '3101'
left join  ActivitiesOrdered as act3102 on act3102.exposurekey = ex.exposurekey and act3102.InteractionTypeKey = '3102'
left join  ActivitiesOrdered as act4019 on act4019.exposurekey = ex.exposurekey and act4019.InteractionTypeKey = '4019'
left join Law4738 on Law4738.cdi=party.cdi 

left join  dexposure dex on dex.exposurekey=eprof.exposurekey and dex.portfoliokey=eprof.portfoliokey and dex.agreementkey=eprof.agreementkey and dex.businessdate='{businessday}'
left join   Restructurings on Restructurings.NewAgreementKey=ex.AgreementKey and restructurings.businessdate='{businessday}'
where party.cdi is not null

   """)
   

second_part.createOrReplaceTempView("second_part")
#display(second_part)


In [0]:
coc_final = spark.sql("""
select * from fisrt_part
union 
select * 
from second_part
""")
coc_final.createOrReplaceTempView("coc_final")
display(coc_final)


checks

In [0]:
#final checks . 
#contracts . this must return none
contact_checks=spark.sql("""
select  distinct ContractNo 
  from coc_final
  except 
  select ContractNo from Contracts
union
  select ContractNo from Contracts
  except 
  select  distinct ContractNo 
  from coc_final
  """)

#display(contact_checks)


export

In [0]:
%pip install openpyxl

In [0]:
try:    
    report_args = dbutils.widgets.get("report_args")
    report_args = report_args.replace("\'", "\"")
except Exception as e:
    raise ValueError("Missing required input: report_args")

# Read JSON
json_data = json.loads(report_args)
 
# Access the values and store them into variables
if "output_file_prefix" not in json_data: raise KeyError("'output_file_prefix' key is missing in the report_args JSON data.")
if "report_output" not in json_data: raise KeyError("'report_output' key is missing in the report_args JSON data.")

output_file_prefix = "".join(json_data['output_file_prefix'])
report_output = json_data['report_output']

In [0]:
from datetime import datetime
from pyspark.sql.functions import col
import os
import pandas as pd
from io import BytesIO

# Define the output directory path
run_date_str = datetime.now().strftime("%Y%m%d")
output_dir = f"{report_output}{run_date_str}/"

# Ensure the directory exists
os.makedirs(output_dir, exist_ok=True)

def export_to_excel(df, sheet_name):
    # Convert columns with dtype 'void' to string
    for col_name, dtype in df.dtypes:
        if dtype == 'void':
            df = df.withColumn(col_name, col(col_name).cast('string'))

    # Use the correct format string for parsing businessday
    output_filename = f"coc_letter_{datetime.strptime(businessday, '%Y-%m-%d').strftime('%Y%m%d')}.xlsx"
    output_path = os.path.join(output_dir, output_filename)

    # Create an Excel writer object
    output_buffer = BytesIO()
    with pd.ExcelWriter(output_buffer, engine='openpyxl') as writer:
        # Save the subset to a temporary CSV file
        temp_file = os.path.join(output_dir, f"coc.csv")
        df.coalesce(1).write.csv(temp_file, header=True, mode="overwrite")
        
        # Read the CSV back into Pandas
        csv_files = [f for f in os.listdir(temp_file) if f.endswith(".csv")]
        if csv_files:
            pandas_df = pd.read_csv(os.path.join(temp_file, csv_files[0]))
            # Write to an Excel sheet
            pandas_df.to_excel(writer, sheet_name=sheet_name, index=False)
        for file in os.listdir(temp_file):
            os.remove(os.path.join(temp_file, file))

    with open(output_path, "wb") as f:
        f.write(output_buffer.getvalue())

In [0]:
export_to_excel(coc_final, 'coc_letter')